<a href="https://colab.research.google.com/github/obiedeh/private-5g-edge-telemetry/blob/main/private_5g_ran_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
"""
Private 5G RAN Pipeline — ingest → transform → parquet

This module provides a research-oriented data engineering pipeline for Private 5G RAN KPIs:
- Ingest raw CSV logs (or generate synthetic RAN logs for experimentation).
- Perform quality checks and feature engineering.
- Aggregate to analysis-ready tables.
- Persist outputs in columnar Parquet format (optionally partitioned by date / cell).

Intended usage:
    python private_5g_ran_pipeline.py \
        --input_dir data/raw \
        --output_parquet data/curated/private5g_ran_kpis.parquet \
        --generate_synthetic

Author: Obinna Edeh
"""

import argparse
import logging
import os
from datetime import datetime, timedelta
from typing import Optional, List

import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# Logging configuration
# -------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("private5g_ran_pipeline")


# -------------------------------------------------------------------
# Synthetic Data Generation (for research & benchmarking)
# -------------------------------------------------------------------
def generate_synthetic_ran_logs(
    n_cells: int = 5,
    n_hours: int = 24,
    freq: str = "5min",
    seed: int = 42,
) -> pd.DataFrame:
    """
    Generate synthetic Private 5G RAN KPI logs.

    Each record represents a (cell_id, timestamp, slice_type) combination with:
      - prb_utilization_dl/ul
      - sinr_db, rsrp_dbm, latency_ms, throughput_mbps
      - ue_count, qos_class_id

    Parameters
    ----------
    n_cells : int
        Number of gNB or small cell sectors.
    n_hours : int
        Number of hours to simulate.
    freq : str
        Frequency string for pandas.date_range (e.g., "5min").
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    DataFrame
        Synthetic RAN KPI dataset.
    """
    np.random.seed(seed)

    start_ts = datetime.utcnow().replace(minute=0, second=0, microsecond=0)
    end_ts = start_ts + timedelta(hours=n_hours)
    timestamps = pd.date_range(start=start_ts, end=end_ts, freq=freq, inclusive="left")

    cell_ids = [f"cell_{i:03d}" for i in range(1, n_cells + 1)]
    slice_types = ["eMBB", "URLLC", "mMTC"]

    records = []

    for cell in cell_ids:
        # simulate busy-hour pattern (heavier load in the middle of the window)
        busy_center = start_ts + timedelta(hours=n_hours / 2)
        for ts in timestamps:
            for slice_type in slice_types:
                # distance from busy center (in hours)
                delta_hours = abs((ts - busy_center).total_seconds()) / 3600.0
                load_factor = np.exp(-0.5 * (delta_hours / 3.0) ** 2)  # bell-shaped

                base_dl = 0.3 + 0.5 * load_factor
                base_ul = 0.2 + 0.4 * load_factor

                # Slice effects: eMBB heavier, URLLC low latency, mMTC low throughput
                if slice_type == "eMBB":
                    dl_util = np.clip(np.random.normal(base_dl, 0.08), 0, 1)
                    ul_util = np.clip(np.random.normal(base_ul, 0.07), 0, 1)
                    latency = np.random.normal(15, 3)
                    throughput = np.random.normal(150 * load_factor, 30)
                    qci = 9  # example QCI for eMBB
                elif slice_type == "URLLC":
                    dl_util = np.clip(np.random.normal(base_dl * 0.7, 0.05), 0, 1)
                    ul_util = np.clip(np.random.normal(base_ul * 0.7, 0.05), 0, 1)
                    latency = np.random.normal(5, 1)  # ultra-low latency
                    throughput = np.random.normal(50 * load_factor, 10)
                    qci = 7  # example
                else:  # mMTC
                    dl_util = np.clip(np.random.normal(base_dl * 0.4, 0.03), 0, 1)
                    ul_util = np.clip(np.random.normal(base_ul * 0.4, 0.03), 0, 1)
                    latency = np.random.normal(25, 5)
                    throughput = np.random.normal(10 * load_factor, 3)
                    qci = 1  # example

                # radio conditions
                sinr = np.random.normal(17 - 5 * load_factor, 3)  # dB
                rsrp = np.random.normal(-90, 4)  # dBm

                ue_count = int(np.clip(np.random.normal(40 * load_factor, 10), 1, 500))

                records.append(
                    {
                        "timestamp": ts,
                        "date": ts.date().isoformat(),
                        "cell_id": cell,
                        "slice_type": slice_type,
                        "prb_utilization_dl": dl_util,
                        "prb_utilization_ul": ul_util,
                        "sinr_db": sinr,
                        "rsrp_dbm": rsrp,
                        "latency_ms": latency,
                        "throughput_mbps": throughput,
                        "ue_count": ue_count,
                        "qos_class_id": qci,
                    }
                )

    df = pd.DataFrame.from_records(records)
    logger.info("Generated synthetic RAN logs: %d rows", len(df))
    return df


# -------------------------------------------------------------------
# Ingestion: read raw logs from CSV directory
# -------------------------------------------------------------------
def ingest_csv_directory(input_dir: str, pattern: str = ".csv") -> pd.DataFrame:
    """
    Ingest raw RAN KPI CSV logs from a directory.

    Assumes that all files share a compatible schema or a superset of columns.

    Parameters
    ----------
    input_dir : str
        Directory path containing CSV files.
    pattern : str
        Filename suffix to filter by (default: '.csv').

    Returns
    -------
    DataFrame
        Concatenated raw logs.
    """
    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")

    csv_files: List[str] = [
        os.path.join(input_dir, f)
        for f in os.listdir(input_dir)
        if f.endswith(pattern)
    ]

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {input_dir} (pattern={pattern})")

    frames = []
    for path in csv_files:
        logger.info("Reading CSV: %s", path)
        df = pd.read_csv(path)
        frames.append(df)

    combined = pd.concat(frames, ignore_index=True)
    logger.info("Ingested %d rows from %d files", len(combined), len(csv_files))
    return combined


# -------------------------------------------------------------------
# Transformations: QC, feature engineering, aggregation
# -------------------------------------------------------------------
def qc_and_cast(df: pd.DataFrame) -> pd.DataFrame:
    """
    Basic quality checks and type casting.

    - Ensures timestamp is datetime.
    - Drops rows with missing critical fields (cell_id, slice_type, timestamp).
    - Clips key KPI metrics to plausible ranges.
    """
    df = df.copy()

    # Ensure timestamp is datetime
    if not np.issubdtype(df["timestamp"].dtype, np.datetime64):
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    # Derive date if not present
    if "date" not in df.columns:
        df["date"] = df["timestamp"].dt.date.astype(str)

    # Drop bad rows
    before = len(df)
    df = df.dropna(subset=["timestamp", "cell_id", "slice_type"])
    df = df[df["timestamp"].notna()]
    after = len(df)
    logger.info("Dropped %d rows with invalid timestamp/cell_id/slice_type", before - after)

    # Clip metrics
    clip_config = {
        "prb_utilization_dl": (0.0, 1.0),
        "prb_utilization_ul": (0.0, 1.0),
        "sinr_db": (-10.0, 40.0),
        "rsrp_dbm": (-140.0, -60.0),
        "latency_ms": (0.0, 1000.0),
        "throughput_mbps": (0.0, 2000.0),
    }

    for col, (lo, hi) in clip_config.items():
        if col in df.columns:
            df[col] = df[col].astype(float)
            df[col] = df[col].clip(lo, hi)

    if "ue_count" in df.columns:
        df["ue_count"] = df["ue_count"].fillna(0).astype(int)
        df["ue_count"] = df["ue_count"].clip(0, 10000)

    return df


def engineer_features(df: pd.DataFrame, carrier_bandwidth_mhz: float = 100.0) -> pd.DataFrame:
    """
    Feature engineering for research / analytics:
    - Spectral efficiency (bps/Hz).
    - Cell load class (low/medium/high).
    - Congestion indicator.

    Parameters
    ----------
    df : DataFrame
        Cleaned KPI dataframe.
    carrier_bandwidth_mhz : float
        Downlink carrier bandwidth in MHz (used for spectral efficiency).

    Returns
    -------
    DataFrame
        KPI dataframe with additional engineered features.
    """
    df = df.copy()

    bw_hz = carrier_bandwidth_mhz * 1e6

    if "throughput_mbps" in df.columns:
        # Convert throughput (Mb/s) to bps/Hz
        df["spectral_eff_bps_per_hz"] = (
            df["throughput_mbps"].astype(float) * 1e6 / bw_hz
        )  # Mb/s → bps; divide by Hz

    # Approximate cell load based on DL PRB utilization
    if "prb_utilization_dl" in df.columns:
        conditions = [
            df["prb_utilization_dl"] < 0.4,
            df["prb_utilization_dl"].between(0.4, 0.7, inclusive="left"),
            df["prb_utilization_dl"] >= 0.7,
        ]
        choices = ["low", "medium", "high"]
        df["cell_load_class"] = np.select(conditions, choices, default="unknown")

    # Congestion indicator: high PRB utilization + elevated latency
    if "prb_utilization_dl" in df.columns and "latency_ms" in df.columns:
        df["is_congested"] = (
            (df["prb_utilization_dl"] >= 0.8) & (df["latency_ms"] >= 30)
        ).astype(int)

    return df


def aggregate_hourly_per_cell_slice(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate 5-minute (or sub-hourly) logs to hourly KPIs per (cell_id, slice_type).

    Computed statistics:
    - mean / p95 latency
    - mean PRB utilization (DL/UL)
    - mean SINR / RSRP
    - sum UE count
    - congestion ratio (% congested samples)
    - mean spectral efficiency

    Returns
    -------
    DataFrame
        Hourly aggregated KPIs.
    """
    df = df.copy()
    df["hour"] = df["timestamp"].dt.floor("H")

    group_cols = ["date", "hour", "cell_id", "slice_type"]

    def p95(x):
        return np.percentile(x, 95)

    agg_dict = {}

    if "latency_ms" in df.columns:
        agg_dict["latency_ms"] = ["mean", p95]
    if "prb_utilization_dl" in df.columns:
        agg_dict["prb_utilization_dl"] = "mean"
    if "prb_utilization_ul" in df.columns:
        agg_dict["prb_utilization_ul"] = "mean"
    if "sinr_db" in df.columns:
        agg_dict["sinr_db"] = "mean"
    if "rsrp_dbm" in df.columns:
        agg_dict["rsrp_dbm"] = "mean"
    if "ue_count" in df.columns:
        agg_dict["ue_count"] = "sum"
    if "is_congested" in df.columns:
        agg_dict["is_congested"] = "mean"
    if "spectral_eff_bps_per_hz" in df.columns:
        agg_dict["spectral_eff_bps_per_hz"] = "mean"

    grouped = df.groupby(group_cols).agg(agg_dict)
    grouped.columns = [
        "_".join(c).replace("<lambda_0>", "p95") if isinstance(c, tuple) else c
        for c in grouped.columns
    ]
    grouped = grouped.reset_index()

    # Rename helpful columns
    col_renames = {
        "latency_ms_mean": "latency_ms_avg",
        "latency_ms_p95": "latency_ms_p95",
        "prb_utilization_dl_mean": "prb_util_dl_avg",
        "prb_utilization_ul_mean": "prb_util_ul_avg",
        "sinr_db_mean": "sinr_db_avg",
        "rsrp_dbm_mean": "rsrp_dbm_avg",
        "ue_count_sum": "ue_count_total",
        "is_congested_mean": "congestion_ratio",
        "spectral_eff_bps_per_hz_mean": "spectral_eff_bps_per_hz_avg",
    }

    grouped = grouped.rename(columns=col_renames)

    logger.info("Aggregated to hourly KPIs: %d rows", len(grouped))
    return grouped


# -------------------------------------------------------------------
# Persistence: write to Parquet
# -------------------------------------------------------------------
def write_parquet(
    df: pd.DataFrame,
    output_path: str,
    partition_cols: Optional[List[str]] = None,
    engine: str = "pyarrow",
) -> None:
    """
    Write DataFrame to Parquet file.

    Parameters
    ----------
    df : DataFrame
        Data to persist.
    output_path : str
        Parquet file path (if partition_cols is None) or directory (if partition_cols set).
    partition_cols : list of str, optional
        Columns to use for partitioning (e.g., ['date', 'cell_id']).
    engine : str
        Parquet engine (e.g., 'pyarrow' or 'fastparquet').
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    logger.info(
        "Writing Parquet to %s (partition_cols=%s, engine=%s)",
        output_path,
        partition_cols,
        engine,
    )

    df.to_parquet(
        output_path,
        engine=engine,
        index=False,
        partition_cols=partition_cols,
    )

    logger.info("Parquet write complete.")


# -------------------------------------------------------------------
# Orchestration: full pipeline
# -------------------------------------------------------------------
def run_pipeline(
    input_dir: Optional[str],
    output_parquet: str,
    generate_synthetic: bool = False,
    partition_cols: Optional[List[str]] = None,
) -> None:
    """
    Run the end-to-end Private 5G RAN pipeline:
        ingest → qc → feature engineering → aggregate → parquet

    Parameters
    ----------
    input_dir : str or None
        Directory containing raw CSV logs. Ignored if generate_synthetic=True.
    output_parquet : str
        Output Parquet file/directory.
    generate_synthetic : bool
        If True, synthetic RAN logs are generated instead of reading CSV files.
    partition_cols : list of str, optional
        Partition columns for Parquet.
    """
    if generate_synthetic:
        logger.info("Using synthetic RAN logs as pipeline input.")
        df_raw = generate_synthetic_ran_logs()
    else:
        if input_dir is None:
            raise ValueError("input_dir must be provided if generate_synthetic is False.")
        df_raw = ingest_csv_directory(input_dir)

    # Transform
    df_clean = qc_and_cast(df_raw)
    df_features = engineer_features(df_clean)
    df_hourly = aggregate_hourly_per_cell_slice(df_features)

    # Persist
    write_parquet(df_hourly, output_parquet, partition_cols=partition_cols)


# -------------------------------------------------------------------
# CLI entrypoint
# -------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Private 5G RAN Pipeline — ingest → transform → parquet",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    parser.add_argument(
        "--input_dir",
        type=str,
        default=None,
        help="Directory containing raw CSV logs. Ignored if --generate_synthetic is set.",
    )
    parser.add_argument(
        "--output_parquet",
        type=str,
        required=True,
        help="Path to output Parquet file or directory.",
    )
    parser.add_argument(
        "--generate_synthetic",
        action="store_true",
        help="Generate synthetic Private 5G RAN logs instead of reading CSV input.",
    )
    parser.add_argument(
        "--partition_by_date_cell",
        action="store_true",
        help="Partition Parquet output by date and cell_id.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()

    if not args.generate_synthetic and args.input_dir is None:
        raise SystemExit(
            "Error: either provide --input_dir or enable --generate_synthetic."
        )

    partition_cols = ["date", "cell_id"] if args.partition_by_date_cell else None

    run_pipeline(
        input_dir=args.input_dir,
        output_parquet=args.output_parquet,
        generate_synthetic=args.generate_synthetic,
        partition_cols=partition_cols,
    )


if __name__ == "__main__":
    main()
